# TexTag: Would more data help? (learning curve)

If we paid for more labeled textures, how much better would the search model get?

**Method:** train the same model on slices of the training data (5, 10, 20 and 40 images per class) and plot the
test score against the number of training images. This is a **learning curve**:

- still climbing at 100% → more data should help
- flat → more data won't help much; spend on something else (better labels, a better model)

A line fitted through the points gives a rough estimate of the gain from 2×, 5× or 50× more images,
which we can put next to the cost of labeling them.

**Model:** the multi-label ResNet18 from `07_search` (same two phases, settings and early stopping).
**Scores:** test mAP and tag-search precision@10, as in `07_search`.

**Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (~25 min).

Sections:
1. Setup
2. Data slices
3. Training function
4. Train on each slice
5. Learning curve
6. Extrapolation
7. Cost vs benefit
8. Findings

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/textag"):
        !git clone -q https://github.com/codecustard/textag.git /content/textag
    %cd /content/textag/notebooks

In [ ]:
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import average_precision_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from textag.model import resnet18, unfreeze
from textag import multilabel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = os.environ.get("TEXTAG_EPOCHS")   # env var only used for quick test runs

## 2. Data slices

Same images and labels as `07_search`. Validation and test stay the same full sets (1,880 images each);
only the **training** set shrinks.

Each slice takes the **same number of images from every class** (by the image's main class), so all 47 classes
are always there and only the amount changes. The slices are **nested**: the 10-per-class slice contains the
5-per-class slice, and so on. So each step only *adds* data.

In [ ]:
dl = multilabel.loaders(batch_size=32, num_workers=2)
train_ds = dl["train"].dataset
classes = train_ds.classes
test_targets = dl["test"].dataset.targets

main_class = train_ds.dtd._labels          # each training image's main class (0-46), from the single-label data

def slice_indices(per_class):
    """The first `per_class` training images of every class."""
    indices = []
    for c in range(len(classes)):
        in_class = [i for i, label in enumerate(main_class) if label == c]
        indices += in_class[:per_class]
    return indices

SLICES = [5, 10, 20, 40]                   # images per class; 40 = the full training set
for per_class in SLICES:
    n = len(slice_indices(per_class))
    print(f"{per_class:>2} per class -> {n:>4} training images ({n / len(train_ds):.1%})")

# nested: every smaller slice is inside the bigger ones
assert set(slice_indices(5)) <= set(slice_indices(10)) <= set(slice_indices(20)) <= set(slice_indices(40))

## 3. Training function

Exactly the `07_search` recipe, wrapped in a function so it can run once per slice:

| | Phase 1: feature extractor | Phase 2: fine-tuning |
|---|---|---|
| What trains | Only the new last layer | All layers |
| Learning rate | 1e-3, up to 15 epochs | 1e-4, up to 20 epochs |
| Early stopping | patience 5 (on the full validation set) | patience 5 |

Loss `BCEWithLogitsLoss`, optimizer `AdamW` (weight decay 1e-4), `ReduceLROnPlateau`. After training,
the model scores the **test** set: mAP (all 47 tags, whole ranking) and tag-search precision@10.

In [ ]:
def train_and_score(per_class):
    """Train the multi-label ResNet18 on one slice; return its test scores."""
    torch.manual_seed(42)                      # same starting point for every slice
    indices = slice_indices(per_class)
    train_dataloader = DataLoader(Subset(train_ds, indices), batch_size=32, shuffle=True, num_workers=2)

    model = resnet18(num_classes=len(classes), pretrained=True, freeze_backbone=True)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    start = timer()

    # Phase 1: only the new last layer
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    hist1 = multilabel.train(model, train_dataloader, dl["val"], optimizer, loss_fn,
                             epochs=int(EPOCHS or 15), device=device, scheduler=scheduler, patience=5, verbose=False)

    # Phase 2: all layers, 10x smaller learning rate
    unfreeze(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    hist2 = multilabel.train(model, train_dataloader, dl["val"], optimizer, loss_fn,
                             epochs=int(EPOCHS or 20), device=device, scheduler=scheduler, patience=5, verbose=False)
    seconds = timer() - start

    # Score the test set
    tag_scores, _ = multilabel.index(model, dl["test"], device)
    return {"per class": per_class,
            "training images": len(indices),
            "test mAP": average_precision_score(test_targets, tag_scores, average="macro"),
            "tag search P@10": multilabel.tag_search_precision(tag_scores, test_targets).mean().item(),
            "epochs (phase 1 + 2)": f"{len(hist1['val_loss'])} + {len(hist2['val_loss'])}",
            "training time (s)": round(seconds)}

## 4. Train on each slice

Four full trainings, smallest first (~25 min on a T4 in total).

In [ ]:
rows = []
for per_class in SLICES:
    row = train_and_score(per_class)
    rows.append(row)
    print(f"{row['training images']:>4} images: test mAP {row['test mAP']:.1%}, "
          f"tag search P@10 {row['tag search P@10']:.1%}  ({row['training time (s)']} s)")

curve = pd.DataFrame(rows).set_index("training images")
curve.style.format({"test mAP": "{:.1%}", "tag search P@10": "{:.1%}"})

## 5. Learning curve

The x-axis is on a **log scale** (each step doubles the data). Learning curves usually look close to a straight
line on this scale, which is what makes the extrapolation in the next section possible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["test mAP", "tag search P@10"]):
    ax.plot(curve.index, curve[metric], "o-", color="#2F5E96")
    for n, value in curve[metric].items():
        ax.annotate(f"{value:.1%}", (n, value), textcoords="offset points", xytext=(0, 8), ha="center")
    ax.set_xscale("log")
    ax.set_xticks(curve.index, [str(n) for n in curve.index])
    ax.set_xlabel("training images (log scale)")
    ax.set_title(metric)
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

for metric in ["test mAP", "tag search P@10"]:
    gains = curve[metric].diff().dropna()
    print(f"{metric}: gain per doubling = " + ", ".join(f"{g:+.1%}" for g in gains))

## 6. Extrapolation

Fit a straight line through the points on the log scale: `score ≈ a + b × log10(images)`.
Then `b` is the gain from **10×** more images, and `b × log10(2)` the gain per **doubling**.

**This is a rough estimate.** The real curve usually bends and flattens with much more data (and can't pass 100%),
so the further we go past 1,880 images, the more optimistic the line becomes. Use it for the order of magnitude only.

In [ ]:
FUTURE = [3_760, 10_000, 100_000]          # 2x DTD's training set, ~5x, ~50x

fits, predictions = {}, {}
for metric in ["test mAP", "tag search P@10"]:
    b, a = np.polyfit(np.log10(curve.index), curve[metric], 1)    # slope, intercept
    fits[metric] = (a, b)
    predictions[metric] = [min(a + b * np.log10(n), 1.0) for n in FUTURE]
    print(f"{metric}: score ≈ {a:.3f} + {b:.3f} × log10(images)  ->  +{b * np.log10(2):.1%} per doubling")

prediction_table = pd.DataFrame(predictions, index=pd.Index(FUTURE, name="training images (extrapolated)"))
prediction_table.style.format("{:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
line_x = np.logspace(np.log10(curve.index.min()), np.log10(FUTURE[-1]), 50)
for ax, metric in zip(axes, ["test mAP", "tag search P@10"]):
    a, b = fits[metric]
    ax.plot(line_x, np.minimum(a + b * np.log10(line_x), 1.0), "--", color="#F0924F", label="fitted line (extrapolated)")
    ax.plot(curve.index, curve[metric], "o", color="#2F5E96", label="measured")
    ax.axvline(curve.index.max(), color="gray", linestyle=":", label="today: 1,880 images")
    ax.set_xscale("log")
    ax.set_xlabel("training images (log scale)")
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Cost vs benefit

Labeling cost for the **extra** images, at an assumed **$0.05–0.10 per image** for simple tag labels
(real quotes vary). GPU cost stays small: our full run takes ~12 min on a T4 (~$0.04), and training time
grows roughly in line with the number of images.

In [ ]:
LABEL_COST = (0.05, 0.10)                  # $ per image, assumption
today = curve.index.max()
gpu_hours_per_image = 12 / 60 / today      # ~12 min on a T4 for 1,880 images (07_search)

rows = []
for i, n in enumerate(FUTURE):
    extra = n - today
    rows.append({"training images": n,
                 "extra images to label": extra,
                 "labeling cost": f"${extra * LABEL_COST[0]:,.0f}–{extra * LABEL_COST[1]:,.0f}",
                 "T4 training time": f"~{n * gpu_hours_per_image:.1f} h",
                 "mAP (est.)": f"{predictions['test mAP'][i]:.0%} ({predictions['test mAP'][i] - curve['test mAP'].iloc[-1]:+.0%})",
                 "tag search P@10 (est.)": f"{predictions['tag search P@10'][i]:.0%} "
                                           f"({predictions['tag search P@10'][i] - curve['tag search P@10'].iloc[-1]:+.0%})"})
pd.DataFrame(rows).set_index("training images")

## 8. Findings

*(Filled in after the full run.)*